# Notebook 1 of 2: CAPTURE-24 preprocessing
**ES 335** · [Nityanand Karmali, 24110226.] · [Amartya Tuljapurkar, 24110034.]

Turns the raw CAPTURE-24 wrist-accelerometer files (151 participants, about 152 GB of CSV, 100 Hz) into one small table with one row per labelled 10-second window. Output: `capture24_features.parquet` and `stats.csv`, which notebook 2 (`capture24-modelling`) reads.

**Data:** CAPTURE-24 (Chan et al., *Scientific Data* 2024; Oxford Research Archive, DOI 10.5287/bodleian:NGx0JOMP5, CC BY 4.0), used through the Kaggle copy `kooaslansefat/capture-24-human-activity-recognition`. The window length, majority-label rule and Walmsley2020 grouping follow the CAPTURE-24 benchmark (github.com/OxWearables/capture24); the code here is our own.

**How to run:** attach the Kaggle dataset, no GPU needed, then **Save Version → Save & Run All** (about 1 hour).

## 1. Setup and paths

In [2]:
import os, glob, time
import numpy as np
import pandas as pd

ROOT = "/kaggle/input/datasets/kooaslansefat/capture-24-human-activity-recognition/capture24"
if not os.path.isdir(ROOT):                 # Kaggle sometimes mounts datasets at a different path
    ROOT = os.path.dirname(glob.glob("/kaggle/input/**/annotation-label-dictionary.csv", recursive=True)[0])
print("data folder:", ROOT)

def pfile(pid):
    # path to one participant's CSV, e.g. pfile('P007')
    return f"{ROOT}/{pid}.csv/{pid}.csv"

PIDS = sorted(x[:-4] for x in os.listdir(ROOT) if x.startswith("P") and x.endswith(".csv"))
print(len(PIDS), PIDS[:3], PIDS[-3:])      # expect 151 ['P001', ...] [..., 'P151']

dic  = pd.read_csv(f"{ROOT}/annotation-label-dictionary.csv")
meta = pd.read_csv(f"{ROOT}/metadata.csv")
print(dic.columns.tolist())
print(meta.columns.tolist())               # expect ['pid', 'age', 'sex']

data folder: /kaggle/input/datasets/kooaslansefat/capture-24-human-activity-recognition/capture24
151 ['P001', 'P002', 'P003'] ['P149', 'P150', 'P151']
['annotation', 'label:WillettsSpecific2018', 'label:WillettsMET2018', 'label:DohertySpecific2018', 'label:Willetts2018', 'label:Doherty2018', 'label:Walmsley2020']
['pid', 'age', 'sex']


## 2. Label grouping (Walmsley2020)
The raw annotations have over 200 fine-grained labels. We use the dataset's own `label:Walmsley2020` grouping into 4 classes: sleep, sedentary, light, moderate-vigorous.

In [3]:
KEY, LABEL = "annotation", "label:Walmsley2020"
label_map = dict(zip(dic[KEY], dic[LABEL]))

print(pd.Series(label_map).value_counts())   # raw labels per group

light                94
sedentary            65
moderate-vigorous    46
sleep                 1
Name: count, dtype: int64


## 3. Processing one participant
- **Window:** 1,000 consecutive rows (10 s at 100 Hz), non-overlapping.
- **Features (16):** mean, std, min and max of x, y, z and the magnitude sqrt(x² + y² + z²).
- **Label:** the most frequent mapped label inside the window. Windows with no label are dropped.
- **Quality stats** are recorded for every participant: identical consecutive samples, expected vs actual rows (gaps), labelled windows, labels missing from the dictionary.

In [4]:
WIN = 1000                                  # 10 s at 100 Hz
OUT = "/kaggle/working/features"
os.makedirs(OUT, exist_ok=True)

def process_participant(pid, label_map):
    df = pd.read_csv(pfile(pid),
                     usecols=["time", "x", "y", "z", "annotation"],
                     dtype={"x": "float32", "y": "float32", "z": "float32",
                            "annotation": str},
                     low_memory=False)

    # data-quality stats for the write-up
    xa, ya, za = (df[c].to_numpy() for c in "xyz")
    same = ((np.diff(xa) == 0) & (np.diff(ya) == 0) & (np.diff(za) == 0)).mean()
    t0, t1 = pd.to_datetime(df["time"].iloc[0]), pd.to_datetime(df["time"].iloc[-1])
    expected_rows = (t1 - t0).total_seconds() * 100 + 1
    unmapped = set(df["annotation"].dropna().unique()) - set(label_map)

    # windowing and features
    cls = df["annotation"].map(label_map)
    n = len(df) // WIN * WIN
    x, y, z = (a[:n].reshape(-1, WIN) for a in (xa, ya, za))
    m = np.sqrt(x**2 + y**2 + z**2)

    out = {}
    for name, a in [("x", x), ("y", y), ("z", z), ("m", m)]:
        out[f"{name}_mean"] = a.mean(1)
        out[f"{name}_std"]  = a.std(1)
        out[f"{name}_min"]  = a.min(1)
        out[f"{name}_max"]  = a.max(1)
    out = pd.DataFrame(out)

    # window label = most frequent label inside the window
    lab = cls.to_numpy()[:n].reshape(-1, WIN)
    out["label"] = [pd.Series(r).dropna().mode().iloc[0] if pd.notna(r).any() else None
                    for r in lab]
    out["pid"] = pid
    n_total = len(out)
    out = out.dropna(subset=["label"])

    stats = dict(pid=pid, rows=len(df), expected_rows=round(expected_rows),
                 share_identical=round(float(same), 3),
                 windows_total=n_total, windows_labelled=len(out),
                 n_unmapped_labels=len(unmapped))
    return out, stats

## 4. Test on 3 participants
Check that `rows` is close to `expected_rows` and that `n_unmapped_labels` is 0.

In [5]:
for pid in PIDS[:3]:
    t = time.time()
    res, st = process_participant(pid, label_map)
    print(st, round(time.time() - t, 1), "s")
    print(res["label"].value_counts().to_dict())

{'pid': 'P001', 'rows': 10020001, 'expected_rows': 10020001, 'share_identical': 0.371, 'windows_total': 10020, 'windows_labelled': 7590, 'n_unmapped_labels': 0} 35.8 s
{'sleep': 2743, 'sedentary': 2659, 'light': 2172, 'moderate-vigorous': 16}
{'pid': 'P002', 'rows': 8820001, 'expected_rows': 8820001, 'share_identical': 0.301, 'windows_total': 8820, 'windows_labelled': 5699, 'n_unmapped_labels': 0} 28.6 s
{'sleep': 2161, 'sedentary': 1954, 'moderate-vigorous': 974, 'light': 610}
{'pid': 'P003', 'rows': 9720001, 'expected_rows': 9720001, 'share_identical': 0.292, 'windows_total': 9720, 'windows_labelled': 6014, 'n_unmapped_labels': 0} 33.1 s
{'sedentary': 2141, 'sleep': 1981, 'light': 1892}


## 5. Full run
Saves each participant as it finishes, so a rerun after a crash skips finished ones.

In [6]:
STATS = "/kaggle/working/stats.csv"

for pid in PIDS:
    f = f"{OUT}/{pid}.parquet"
    if os.path.exists(f):                  # skip finished ones after a crash
        continue
    t = time.time()
    res, st = process_participant(pid, label_map)
    res.to_parquet(f, index=False)
    pd.DataFrame([st]).to_csv(STATS, mode="a", index=False,
                              header=not os.path.exists(STATS))
    print(pid, res.shape, round(time.time() - t, 1), "s")

done = sorted(os.path.basename(f)[:-8] for f in glob.glob(OUT + "/*.parquet"))
print(len(done), "of", len(PIDS))
print("missing:", sorted(set(PIDS) - set(done)))

P001 (7590, 18) 31.4 s
P002 (5699, 18) 25.6 s
P003 (6014, 18) 29.1 s
P004 (5687, 18) 28.5 s
P005 (7056, 18) 36.6 s
P006 (8211, 18) 47.2 s
P007 (7719, 18) 37.8 s
P008 (6747, 18) 34.1 s
P009 (2328, 18) 23.3 s
P010 (6405, 18) 36.0 s
P011 (6109, 18) 31.7 s
P012 (6962, 18) 32.3 s
P013 (6358, 18) 32.2 s
P014 (7079, 18) 34.0 s
P015 (5600, 18) 31.0 s
P016 (5965, 18) 31.3 s
P017 (6898, 18) 34.3 s
P018 (4059, 18) 27.1 s
P019 (6875, 18) 32.9 s
P020 (4846, 18) 27.8 s
P021 (6518, 18) 31.9 s
P022 (8792, 18) 37.9 s
P023 (5619, 18) 30.8 s
P024 (7425, 18) 48.8 s
P025 (7161, 18) 40.5 s
P026 (6045, 18) 32.5 s
P027 (6416, 18) 37.1 s
P028 (4986, 18) 32.3 s
P029 (5981, 18) 37.5 s
P030 (7813, 18) 39.1 s
P031 (5572, 18) 33.6 s
P032 (7127, 18) 35.0 s
P033 (7687, 18) 39.3 s
P034 (4770, 18) 36.5 s
P035 (6876, 18) 37.3 s
P036 (5445, 18) 31.1 s
P037 (6313, 18) 34.2 s
P038 (6688, 18) 37.5 s
P039 (5233, 18) 33.9 s
P040 (6259, 18) 34.2 s
P041 (5917, 18) 50.8 s
P042 (6319, 18) 40.0 s
P043 (6753, 18) 37.8 s
P044 (4922,

## 6. Combine into one table

In [7]:
feat = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(OUT + "/*.parquet"))],
                 ignore_index=True)
feat = feat.merge(meta, on="pid", how="left")
feat.to_parquet("/kaggle/working/capture24_features.parquet", index=False)

print(feat.shape)
print(feat["label"].value_counts())
print(feat.groupby("pid").size().describe())

(934776, 20)
label
sedentary            371087
sleep                340231
light                178010
moderate-vigorous     45448
Name: count, dtype: int64
count     151.000000
mean     6190.569536
std      1195.929817
min      2214.000000
25%      5478.000000
50%      6259.000000
75%      7007.000000
max      9009.000000
dtype: float64


## 7. Class counts in the train and test participants
P001–P101 train, P102–P151 test (CAPTURE-24 benchmark split).

In [8]:
num = feat["pid"].str[1:].astype(int)
feat["split"] = np.where(num <= 101, "train", "test")

print(pd.crosstab(feat["label"], feat["split"]))
print(pd.crosstab(feat["label"], feat["split"], normalize="columns").round(4))

split                test   train
label                            
light               62850  115160
moderate-vigorous   17807   27641
sedentary          112804  258283
sleep              107890  232341
split                test   train
label                            
light              0.2086  0.1818
moderate-vigorous  0.0591  0.0436
sedentary          0.3743  0.4078
sleep              0.3580  0.3668


## 8. Data-quality summary

In [9]:
st = pd.read_csv(STATS)
st["unlabelled_share"] = 1 - st["windows_labelled"] / st["windows_total"]
print(st[["share_identical", "unlabelled_share"]].describe())
print("participants with unmapped labels:", (st["n_unmapped_labels"] > 0).sum())

st["row_diff_pct"] = 100 * (st["rows"] - st["expected_rows"]) / st["expected_rows"]
bad = st[st["row_diff_pct"].abs() > 1][["pid", "rows", "expected_rows", "row_diff_pct", "windows_labelled"]]
print(bad.sort_values("row_diff_pct"))
ids = bad["pid"].str[1:].astype(int)
print("in train:", (ids <= 101).sum(), "| in test:", (ids > 101).sum())

mv = feat["label"].eq("moderate-vigorous").groupby(feat["pid"]).mean()
print(mv.describe().round(4))
print((mv < 0.005).sum(), "participants have under 0.5% moderate-vigorous windows")

       share_identical  unlabelled_share
count       151.000000        151.000000
mean          0.334278          0.332199
std           0.039816          0.105893
min           0.231000          0.156461
25%           0.307500          0.261006
50%           0.330000          0.309853
75%           0.361000          0.400504
max           0.438000          0.681401
participants with unmapped labels: 0
      pid      rows  expected_rows  row_diff_pct  windows_labelled
93   P094   5773541        6675465    -13.511029              2214
96   P097  10180494       11574001    -12.039976              7277
141  P142   5959687        6650240    -10.383881              4001
59   P060   9144258        9982157     -8.393967              5090
76   P077   8055045        8623999     -6.597334              4936
7    P008   9585588       10260001     -6.573225              6747
15   P016   8874001        9234001     -3.898635              5965
16   P017   9720001       10080001     -3.571428          